# Multi-Agent
# 0. 介绍

**研究背景**：复杂任务往往包含可以并行处理的子任务，也需要不同能力分别负责规划、执行和验证。单个 Agent 的上下文、时间和专业能力有限，因此外层程序需要决定如何拆分任务、把子任务交给谁、怎样传递必要信息，以及何时才能确认整体任务完成。

**现存问题**：生产中常见的错误基线是让多个 Agent 共用一段不断增长的对话，通过自由文本轮流分工，并在第一个 Agent 返回结果后就把整个任务标记为完成。真实系统中由此出现过重复执行、任务无人负责、错误结果向下游传播、交接信息丢失、循环对话和漏掉最终验证等问题；参与者越多，上下文、Token、延迟和故障传播范围通常也越大，多 Agent 并不会自动比单 Agent 更可靠。

**解决方案**：本 Notebook 将实现一个极简的 Multi-Agent，采用`分层 Orchestrator-Worker + 显式任务状态 + 结构化 Handoff + 独立验证关卡`机制：由协调者把目标拆成带负责人、依赖、预算和完成条件的子任务，按依赖关系将任务分发给上下文相互隔离的执行者，只传递完成当前工作所需的信息，并用结构化产物关联每次交接；最后由独立验证者依据环境证据检查所有子任务，只有任务看板全部完成且验证通过，整体流程才能停止。然后使用同一份真实 API 任务进行对比：基线版本收到首个局部结果后提前结束; 改进版本完成规划、执行、交接和验证闭环，从而直观看到多 Agent 的关键不是“增加几个模型角色”，而是让外层 Harness 明确管理任务所有权、信息边界和完成条件。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备

## 2.1 固定发布决策任务

多 Agent 适合处理一个角色无法独立覆盖的任务。下面固定一个生产发布决策：业务指标支持上线，但可靠性指标可能阻止上线，最终答复必须同时考虑两方面。

In [2]:
# request 固定所有执行路径共同处理的用户目标
# decision_options 限制最终决定只有上线或暂缓
release_task = {
    "product": "checkout-v2",
    "request": "请评估 checkout-v2 是否可以全量上线。答复必须同时包含业务收益、可靠性风险和最终决定。",
    "decision_options": ["launch", "hold"],
}

print("产品：", release_task["product"])
print("任务：", release_task["request"])
print("可选决定：", release_task["decision_options"])

产品： checkout-v2
任务： 请评估 checkout-v2 是否可以全量上线。答复必须同时包含业务收益、可靠性风险和最终决定。
可选决定： ['launch', 'hold']


输出给出了唯一任务和两个可选决定。业务收益与可靠性风险缺一不可，因此任何单个专家的局部结论都不能代表整体任务完成；下一步准备两个专家各自需要读取的材料。

## 2.2 准备相互制约的材料

真实发布评审通常需要汇总不同团队的证据。下面准备业务实验与可靠性灰度报告：前者建议上线，后者因为错误率超过门槛而建议暂缓。

In [3]:
# business 只包含业务实验结果，不包含可靠性数据
# reliability 只包含灰度错误率和发布门槛
source_documents = {
    "business": {
        "owner": "business_analyst",
        "content": "灰度实验显示 checkout-v2 的支付转化率提升 8.0%，业务侧建议上线。",
    },
    "reliability": {
        "owner": "reliability_analyst",
        "content": "灰度期间错误率为 3.2%，发布门槛要求不高于 1.0%，可靠性侧建议暂缓。",
    },
}

for source_name, document in source_documents.items():
    print(source_name, "|", document["owner"], "|", document["content"])

business | business_analyst | 灰度实验显示 checkout-v2 的支付转化率提升 8.0%，业务侧建议上线。
reliability | reliability_analyst | 灰度期间错误率为 3.2%，发布门槛要求不高于 1.0%，可靠性侧建议暂缓。


输出显示两份材料给出不同的局部建议。业务分析师只看到转化率，可靠性分析师只看到错误率；这种信息边界能让每个角色专注自己的证据，也意味着协调者必须等待两份回执。

## 2.3 明确角色职责

多 Agent 不是让多个模型自由聊天，而是让每个角色承担明确工作。下面固定两个分析角色和一个审阅角色，后续基线与改进版本都使用相同分工。

In [4]:
# 两个 analyst 分别读取自己负责的唯一材料
# reviewer 只汇总结构化回执并形成最终决定
agent_jobs = [
    {"role": "business_analyst", "input": "business", "job": "提取业务收益并给出局部建议"},
    {"role": "reliability_analyst", "input": "reliability", "job": "比较错误率与门槛并给出局部建议"},
    {"role": "reviewer", "input": "两个分析回执", "job": "汇总证据并给出最终决定"},
]

for agent_job in agent_jobs:
    print(agent_job["role"], "<-", agent_job["input"], "->", agent_job["job"])

business_analyst <- business -> 提取业务收益并给出局部建议
reliability_analyst <- reliability -> 比较错误率与门槛并给出局部建议
reviewer <- 两个分析回执 -> 汇总证据并给出最终决定


输出展示了清晰的责任链：两个分析师产生局部证据，审阅者消费两份回执并负责最终决定。角色已经固定，下一步统一模型提交结果的格式。

## 2.4 定义结构化回执

自由文本交接容易遗漏来源或结论。下面用两个工具格式约束回执：分析师必须提交来源、发现和局部建议，审阅者必须提交两类发现与最终决定。

In [5]:
# analysis_tool 统一两个分析师交回的三个字段
# decision_tool 要求审阅者同时保留两类证据
analysis_tool = {
    "type": "function",
    "function": {
        "name": "submit_analysis",
        "description": "提交一名分析师的局部结论",
        "parameters": {
            "type": "object",
            "properties": {
                "source": {"type": "string", "enum": ["business", "reliability"]},
                "finding": {"type": "string"},
                "recommendation": {"type": "string", "enum": ["launch", "hold"]},
            },
            "required": ["source", "finding", "recommendation"],
        },
    },
}

decision_tool = {
    "type": "function",
    "function": {
        "name": "submit_release_decision",
        "description": "提交汇总后的发布决定",
        "parameters": {
            "type": "object",
            "properties": {
                "business_finding": {"type": "string"},
                "reliability_finding": {"type": "string"},
                "decision": {"type": "string", "enum": ["launch", "hold"]},
            },
            "required": ["business_finding", "reliability_finding", "decision"],
        },
    },
}

print("分析回执：", analysis_tool["function"]["parameters"]["required"])
print("最终回执：", decision_tool["function"]["parameters"]["required"])

分析回执： ['source', 'finding', 'recommendation']
最终回执： ['business_finding', 'reliability_finding', 'decision']


输出列出了两种回执的必填字段。结构化格式只负责让交接内容明确可读，不负责决定何时结束流程；下一步固定所有执行路径共同使用的成功标准。

## 2.5 定义成功标准

可靠性错误率超过发布门槛，因此唯一正确决定是暂缓。最终结果还必须保留业务提升和可靠性错误率，才能证明两个专家的工作都进入了整体结论。

In [6]:
# required_sources 表示整体任务必须收齐的两份回执
# expected_decision 由错误率超过发布门槛这一事实决定
expected = {
    "required_sources": ["business", "reliability"],
    "business_signal": "8.0%",
    "reliability_signal": "3.2%",
    "expected_decision": "hold",
}

print("必须收齐：", expected["required_sources"])
print("必须保留：", expected["business_signal"], "和", expected["reliability_signal"])
print("正确决定：", expected["expected_decision"])

必须收齐： ['business', 'reliability']
必须保留： 8.0% 和 3.2%
正确决定： hold


输出给出了唯一成功标准。至此，发布任务、两份材料、三个角色、结构化回执和正确决定都已固定；下一章将让两个分析角色分别调用真实 API，并保存它们的局部结果。

# 3. 获取并验证 API 响应

## 3.1 为两个分析师准备隔离消息

每个分析师只应看到完成自己工作所需的材料。下面分别建立两组消息：两个子任务服务于同一个发布决策，但业务分析师只分析业务材料，可靠性分析师只分析可靠性材料。

In [7]:
# 每个角色使用独立消息列表，避免共享不断增长的对话
# source_name 决定当前角色唯一能够看到的材料
analyst_messages = {}

for agent_job in agent_jobs[:2]:
    role = agent_job["role"]
    source_name = agent_job["input"]
    document = source_documents[source_name]["content"]
    analyst_messages[role] = [
        {
            "role": "system",
            "content": f"你是 {role}。只完成分配给你的局部分析，不做整体发布决定。调用 submit_analysis，source 必须是 {source_name}。recommendation 只表示当前领域的建议。",
        },
        {
            "role": "user",
            "content": f"子任务：{agent_job['job']}\n当前材料：{document}",
        },
    ]

for role, messages in analyst_messages.items():
    print("角色：", role)
    print(messages[-1]["content"])

角色： business_analyst
子任务：提取业务收益并给出局部建议
当前材料：灰度实验显示 checkout-v2 的支付转化率提升 8.0%，业务侧建议上线。
角色： reliability_analyst
子任务：比较错误率与门槛并给出局部建议
当前材料：灰度期间错误率为 3.2%，发布门槛要求不高于 1.0%，可靠性侧建议暂缓。


输出展示了两个彼此隔离的输入上下文。每个角色只拿到自己的子任务和材料，不会替协调者猜测缺失信息或做整体决定；下一步定义它们共用的真实 API 请求。

## 3.2 定义统一的真实 API 请求

两个分析师应使用完全相同的模型参数，差别只来自角色和材料。下面定义一个极简请求函数，发送消息、读取结构化工具参数，并记录真实 Token、延迟和停止原因。

In [8]:
import json
from time import perf_counter


# messages 是一名分析师的独立上下文
# 返回值同时保留模型决定和本次真实调用指标
def ask_analyst(messages):
    started_at = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=[analysis_tool],
        tool_choice="required",
        temperature=0,
    )
    latency_ms = round((perf_counter() - started_at) * 1000)

    tool_call = response.choices[0].message.tool_calls[0]
    result = json.loads(tool_call.function.arguments)
    usage = response.usage
    metrics = {
        "provider": config["NANO_BACKEND"],
        "model": model_name,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "cost_usd": None,
        "latency_ms": latency_ms,
        "stop_reason": response.choices[0].finish_reason,
    }
    return result, metrics


print("真实分析请求已定义")

真实分析请求已定义


输出说明统一请求函数已经定义，此时仍未发送模型请求。它不负责路由或判断整体任务是否完成，只负责取得一名分析师的回执；下一步调用业务分析师。

## 3.3 获取业务分析师回执

业务分析师只读取转化率材料。下面发送第一次真实 API 请求，并同时打印模型提交的局部发现、建议和本次调用指标。

In [9]:
# 输入只使用业务分析师在 3.1 节得到的消息
# 结果与指标分别保存，供后续路由和消融对照使用
business_result, business_metrics = ask_analyst(
    analyst_messages["business_analyst"]
)

print("业务回执：", business_result)
print("调用指标：", business_metrics)

业务回执： {'source': 'business', 'finding': '灰度实验显示 checkout-v2 的支付转化率提升 8.0%，表明新版本能显著提升用户完成支付的比例，直接带来收入增长', 'recommendation': 'launch'}
调用指标： {'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 258, 'output_tokens': 213, 'total_tokens': 471, 'cost_usd': None, 'latency_ms': 6194, 'stop_reason': 'tool_calls'}


输出显示业务分析师提取了 `8.0%` 的转化率提升，并依据自己可见的材料建议上线。这个建议只代表一个局部视角，不能直接作为整体决定；下一步取得可靠性分析师的独立回执。

## 3.4 获取可靠性分析师回执

可靠性分析师只读取错误率和发布门槛。下面发送第二次真实 API 请求，使用与业务分析师完全相同的模型和采样参数。

In [10]:
# 输入只使用可靠性分析师在 3.1 节得到的消息
# 独立请求保证业务回执不会进入当前角色的上下文
reliability_result, reliability_metrics = ask_analyst(
    analyst_messages["reliability_analyst"]
)

print("可靠性回执：", reliability_result)
print("调用指标：", reliability_metrics)

可靠性回执： {'source': 'reliability', 'finding': '灰度期间错误率为 3.2%，超出发布门槛要求的 1.0%，存在显著可靠性风险', 'recommendation': 'hold'}
调用指标： {'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 265, 'output_tokens': 258, 'total_tokens': 523, 'cost_usd': None, 'latency_ms': 6607, 'stop_reason': 'tool_calls'}


输出显示可靠性分析师比较了 `3.2%` 错误率与 `1.0%` 门槛，并建议暂缓上线。现在两个局部结论已经齐全，下一步按真实到达顺序保存 handoff。

## 3.5 保存专家 Handoff

Router 接收的是逐个到达的回执，而不是自动形成的整体答案。下面按本次真实调用顺序保存两份结构化 handoff，业务回执排在第一，可靠性回执排在第二。

In [11]:
# 列表顺序表示 Router 实际收到回执的先后顺序
# 每份 handoff 保留来源、发现和局部建议三个字段
specialist_handoffs = [
    business_result,
    reliability_result,
]

for position, handoff in enumerate(specialist_handoffs, start=1):
    print(position, handoff["source"], "->", handoff["recommendation"])

1 business -> launch
2 reliability -> hold


输出展示了本次真实运行的 handoff 到达顺序：支持上线的业务回执先到，要求暂缓的可靠性回执后到。两份回执都已保存，但尚未决定 Router 应等待全部结果还是收到第一份就结束；下一章只定义生产中常见的错误基线组件。

# 4. 定义基线组件

## 定义 First-Response-Wins Router

生产中的异步编排常把“一个子任务完成”误写成“根任务完成”。下面复现这个错误 Router：它只读取最先到达的 handoff，把该专家的局部建议直接当成最终决定，然后立即把整体状态标记为完成。

In [12]:
# handoffs 按 Router 实际收到回执的顺序排列
# 第一份局部回执会被错误地提升为整体结果
def first_response_router(handoffs):
    first_handoff = handoffs[0]
    return {
        "status": "completed",
        "received_sources": [first_handoff["source"]],
        "finding": first_handoff["finding"],
        "decision": first_handoff["recommendation"],
    }


print("基线流程：收到第一份 handoff -> 复制局部建议 -> 标记 completed")

基线流程：收到第一份 handoff -> 复制局部建议 -> 标记 completed


输出说明基线 Router 已定义，但还没有处理任何回执。它的错误很明确：`completed` 只由第一份 handoff 触发，与其余子任务是否完成无关；下一章将把第 3 章的真实回执交给它，观察提前结束造成的结果。

# 5. 展示基线故障

## 5.1 运行错误 Router

现在把第 3 章按真实到达顺序保存的两份 handoff 交给基线 Router。业务回执排在第一，因此错误停止规则会在读取可靠性回执之前就结束整体任务。

In [13]:
# 输入是第 3 章保存的真实专家回执列表
# Router 只会读取列表中的第一份业务回执
baseline_result = first_response_router(specialist_handoffs)

print("整体状态：", baseline_result["status"])
print("已收来源：", baseline_result["received_sources"])
print("最终决定：", baseline_result["decision"])
print("采用发现：", baseline_result["finding"])

整体状态： completed
已收来源： ['business']
最终决定： launch
采用发现： 灰度实验显示 checkout-v2 的支付转化率提升 8.0%，表明新版本能显著提升用户完成支付的比例，直接带来收入增长


输出显示根任务已经被标记为 `completed`，但 Router 只接收了业务来源，并把局部 `launch` 建议写成最终决定。可靠性 Agent 已经产出 `hold` 回执，Harness 却没有让它进入结果；下一步用第 2 章的统一标准评分。

## 5.2 判断任务是否真的完成

状态名称不能证明任务成功。下面直接比较已收来源与必需来源，再比较基线决定与正确决定；两项同时满足，整体任务才算通过。

In [14]:
# missing_sources 找出没有进入最终结果的专家来源
# passed 同时要求证据齐全且最终决定正确
missing_sources = []

for required_source in expected["required_sources"]:
    if required_source not in baseline_result["received_sources"]:
        missing_sources.append(required_source)

decision_correct = baseline_result["decision"] == expected["expected_decision"]
baseline_passed = len(missing_sources) == 0 and decision_correct

print("缺失来源：", missing_sources)
print("决定正确：", decision_correct)
print("任务通过：", baseline_passed)

缺失来源： ['reliability']
决定正确： False
任务通过： False


输出中的缺失来源是 `reliability`，决定也不正确，因此任务没有通过。`completed` 只是 Router 过早写入的状态，不是真实完成证据；下一步把这条故障链按发生顺序展开。

## 5.3 展开故障控制流

最终错误来自一个很短的因果链。下面依次打印首个回执到达、根任务提前完成和后续回执被忽略三个事件，直接定位 Harness 的错误停止条件。

In [15]:
# trace 按事件发生顺序记录 Router 的状态变化
# 第三步说明可靠性结果存在，但没有进入整体决定
baseline_trace = [
    {
        "step": 1,
        "event": "handoff_received",
        "source": specialist_handoffs[0]["source"],
        "root_status": "running",
    },
    {
        "step": 2,
        "event": "root_completed",
        "source": specialist_handoffs[0]["source"],
        "root_status": "completed",
    },
    {
        "step": 3,
        "event": "handoff_ignored",
        "source": specialist_handoffs[1]["source"],
        "root_status": "completed",
    },
]

for event in baseline_trace:
    print(event)

{'step': 1, 'event': 'handoff_received', 'source': 'business', 'root_status': 'running'}
{'step': 2, 'event': 'root_completed', 'source': 'business', 'root_status': 'completed'}
{'step': 3, 'event': 'handoff_ignored', 'source': 'reliability', 'root_status': 'completed'}


输出证明故障发生在第二步：业务 Agent 的局部回执触发了根任务完成，随后到达的可靠性回执只能被忽略。两个真实 Agent 都完成了自己的工作，错误来自 Harness 把“任一子任务完成”当成“所有必需子任务完成”；下一章将只定义改进组件。

# 6. 定义改进组件

## 6.1 定义带完成屏障的 Router

可靠编排不能由第一份回执决定结束。下面定义一个 Wait-All Router：它为每个必需来源建立任务状态，逐份收集 handoff，只有所有来源都变成 `completed`，才把 Reviewer 从 `blocked` 改为 `ready`。

In [16]:
# task_board 显式记录每个子任务和 Reviewer 的状态
# review_ready 只有在全部必需来源完成后才会变成 True
def wait_all_router(handoffs, required_sources):
    task_board = {}
    collected = {}

    for source in required_sources:
        task_board[source] = "pending"
    task_board["reviewer"] = "blocked"

    for handoff in handoffs:
        source = handoff["source"]
        collected[source] = handoff
        task_board[source] = "completed"

    review_ready = True
    for source in required_sources:
        if task_board[source] != "completed":
            review_ready = False

    if review_ready:
        task_board["reviewer"] = "ready"

    return {
        "task_board": task_board,
        "collected": collected,
        "review_ready": review_ready,
    }


print("Wait-All Router 已定义：全部来源完成 -> Reviewer ready")

Wait-All Router 已定义：全部来源完成 -> Reviewer ready


输出说明新的 Router 已定义，但尚未处理回执。它把“是否可以进入审阅”变成任务看板上的明确状态，不再依赖哪份回执先到；下一步定义独立 Reviewer。

## 6.2 定义独立 Reviewer

收齐回执不等于已经形成整体结论。下面定义 Reviewer 请求：它只接收两个分析师的结构化 handoff，不读取它们的完整对话，并通过第 2 章的最终回执格式提交发布决定。

In [17]:
# collected 是 Wait-All Router 收齐的结构化回执
# Reviewer 的真实调用继续记录 Token、延迟和停止原因
def ask_reviewer(collected):
    visible_handoffs = [
        collected["business"],
        collected["reliability"],
    ]
    messages = [
        {
            "role": "system",
            "content": "你是发布 Reviewer。必须同时使用两份回执；可靠性门槛未通过时，最终决定必须是 hold。调用 submit_release_decision。",
        },
        {
            "role": "user",
            "content": json.dumps(visible_handoffs, ensure_ascii=False),
        },
    ]

    started_at = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=[decision_tool],
        tool_choice="required",
        temperature=0,
    )
    latency_ms = round((perf_counter() - started_at) * 1000)

    tool_call = response.choices[0].message.tool_calls[0]
    result = json.loads(tool_call.function.arguments)
    usage = response.usage
    metrics = {
        "provider": config["NANO_BACKEND"],
        "model": model_name,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "cost_usd": None,
        "latency_ms": latency_ms,
        "stop_reason": response.choices[0].finish_reason,
    }
    return result, metrics


print("独立 Reviewer 已定义：两份 handoff -> 一个整体决定")

独立 Reviewer 已定义：两份 handoff -> 一个整体决定


输出说明 Reviewer 请求已经定义，此时没有新增 API 调用。Reviewer 只负责综合完整证据，不负责修改任务状态；下一步定义最后的完成关卡。

## 6.3 定义 Reviewer Gate

根任务必须在 Reviewer 返回之后才能完成。下面定义最后一道关卡：它把 Reviewer 状态改成 `completed`，并把两份来源、两类发现和整体决定一起写入最终结果。

In [18]:
# routing_state 提供已经收齐的来源和当前任务看板
# review_result 提供独立 Reviewer 生成的整体结论
def reviewer_gate(routing_state, review_result):
    task_board = routing_state["task_board"].copy()
    task_board["reviewer"] = "completed"

    return {
        "status": "completed",
        "received_sources": list(routing_state["collected"].keys()),
        "business_finding": review_result["business_finding"],
        "reliability_finding": review_result["reliability_finding"],
        "decision": review_result["decision"],
        "task_board": task_board,
    }


print("Reviewer Gate 已定义：Reviewer 返回 -> 根任务 completed")

Reviewer Gate 已定义：Reviewer 返回 -> 根任务 completed


输出说明改进主线的三个组件都已定义，但尚未运行：Router 等齐必需来源，Reviewer 综合隔离的结构化回执，Gate 最后关闭根任务。下一章将使用第 3 章相同的真实专家结果运行这条完整链路。

# 7. 展示修复结果

## 7.1 等待全部专家回执

先把与基线完全相同的两份真实 handoff 交给 Wait-All Router。它会更新任务看板，但不会直接生成最终决定；只有业务与可靠性任务都完成，Reviewer 才能进入 `ready`。

In [19]:
# 输入复用第 3 章保存的两份真实专家回执
# 必需来源复用第 2 章固定的同一成功标准
fixed_routing = wait_all_router(
    specialist_handoffs,
    expected["required_sources"],
)

print("任务看板：", fixed_routing["task_board"])
print("已收来源：", list(fixed_routing["collected"].keys()))
print("可以审阅：", fixed_routing["review_ready"])

任务看板： {'business': 'completed', 'reliability': 'completed', 'reviewer': 'ready'}
已收来源： ['business', 'reliability']
可以审阅： True


输出显示业务与可靠性任务都已完成，Reviewer 才从 `blocked` 变成 `ready`。此时根任务仍未关闭，两份结构化回执将一起进入独立审阅。

## 7.2 获取 Reviewer 的整体决定

现在调用真实 Reviewer。它同时看到业务与可靠性 handoff，需要保留两类关键事实，并在可靠性门槛未通过时给出 `hold`。

In [20]:
# Reviewer 只读取 Router 收齐的结构化 handoff
# 结果和真实调用指标分别保存供后续对照使用
review_result, review_metrics = ask_reviewer(
    fixed_routing["collected"]
)

print("Reviewer 回执：", review_result)
print("调用指标：", review_metrics)

Reviewer 回执： {'business_finding': '灰度实验显示 checkout-v2 的支付转化率提升 8.0%，表明新版本能显著提升用户完成支付的比例，直接带来收入增长', 'reliability_finding': '灰度期间错误率为 3.2%，超出发布门槛要求的 1.0%，存在显著可靠性风险', 'decision': 'hold'}
调用指标： {'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 306, 'output_tokens': 312, 'total_tokens': 618, 'cost_usd': None, 'latency_ms': 7482, 'stop_reason': 'tool_calls'}


输出显示真实 Reviewer 同时保留了 `8.0%` 业务提升与 `3.2%` 可靠性错误率，并给出 `hold`。模型已经完成综合判断，但根任务还要经过明确的 Reviewer Gate。

## 7.3 通过 Reviewer Gate

Reviewer 回执到达后，Harness 才允许关闭根任务。下面把完整路由状态与 Reviewer 结果交给 Gate，生成包含任务看板和两类证据的最终结果。

In [21]:
# Gate 接收完整路由状态和 Reviewer 的整体结论
# 根任务只在这一刻被标记为 completed
fixed_result = reviewer_gate(fixed_routing, review_result)

print("整体状态：", fixed_result["status"])
print("最终决定：", fixed_result["decision"])
print("任务看板：", fixed_result["task_board"])

整体状态： completed
最终决定： hold
任务看板： {'business': 'completed', 'reliability': 'completed', 'reviewer': 'completed'}


输出显示三个角色都已完成，根任务此时才变成 `completed`，最终决定为 `hold`。与基线不同，状态完成和证据完成发生在同一个时刻；下一步用同一标准评分。

## 7.4 判断修复结果

下面继续使用第 2 章的成功标准，不为改进版本放宽条件。最终结果必须收齐两个来源、保留两个关键数值，并给出正确决定，才算通过。

In [22]:
# missing_sources 检查两名分析师是否都进入最终结果
# signals_preserved 检查 Reviewer 是否保留两项关键证据
fixed_missing_sources = []

for required_source in expected["required_sources"]:
    if required_source not in fixed_result["received_sources"]:
        fixed_missing_sources.append(required_source)

fixed_findings = fixed_result["business_finding"] + fixed_result["reliability_finding"]
signals_preserved = expected["business_signal"] in fixed_findings
signals_preserved = signals_preserved and expected["reliability_signal"] in fixed_findings
fixed_decision_correct = fixed_result["decision"] == expected["expected_decision"]
fixed_passed = len(fixed_missing_sources) == 0 and signals_preserved and fixed_decision_correct

print("缺失来源：", fixed_missing_sources)
print("关键证据保留：", signals_preserved)
print("决定正确：", fixed_decision_correct)
print("任务通过：", fixed_passed)

缺失来源： []
关键证据保留： True
决定正确： True
任务通过： True


输出中的缺失来源为空，两个关键数值均被保留，最终决定正确，因此任务通过。修复没有更换专家模型或输入材料，只改变了外层编排的等待条件与完成关卡；最后查看完整控制流。

## 7.5 展开修复控制流

下面把修复后的三次关键状态变化按顺序打印出来：先收齐 handoff，再完成 Reviewer，最后关闭根任务。

In [23]:
# trace 展示完整证据如何逐步到达最终结果
# 根任务在 Reviewer 完成前始终保持 running
fixed_trace = [
    {
        "step": 1,
        "event": "all_handoffs_collected",
        "sources": fixed_result["received_sources"],
        "root_status": "running",
    },
    {
        "step": 2,
        "event": "reviewer_completed",
        "decision": review_result["decision"],
        "root_status": "running",
    },
    {
        "step": 3,
        "event": "root_completed",
        "decision": fixed_result["decision"],
        "root_status": fixed_result["status"],
    },
]

for event in fixed_trace:
    print(event)

{'step': 1, 'event': 'all_handoffs_collected', 'sources': ['business', 'reliability'], 'root_status': 'running'}
{'step': 2, 'event': 'reviewer_completed', 'decision': 'hold', 'root_status': 'running'}
{'step': 3, 'event': 'root_completed', 'decision': 'hold', 'root_status': 'completed'}


输出展示了修复后的完整证据链：两份 handoff 先汇合，Reviewer 再形成整体决定，根任务最后才完成。下一章将把基线与改进版本的成功率、Token、延迟和角色完成情况放在一起比较。

# 8. 汇总消融对照

## 8.1 对比两条执行路径

两条路径使用同一个模型、同一项任务和同一批真实 Analyst 回执。基线虽然调用了两名 Analyst，却只采用第一份回执；改进版本收齐两份回执后再增加一次 Reviewer 调用。下面汇总实际 Token、顺序执行等待时间、来源利用情况和任务结果。

In [24]:
# Analyst 指标来自第 3 章的两次真实 API 调用
# 改进版本在相同 Analyst 成本上再加入 Reviewer 指标
analyst_total_tokens = business_metrics["total_tokens"] + reliability_metrics["total_tokens"]
analyst_total_latency_ms = business_metrics["latency_ms"] + reliability_metrics["latency_ms"]

ablation_rows = [
    {
        "variant": "错误基线",
        "sources_used": len(baseline_result["received_sources"]),
        "api_calls": 2,
        "total_tokens": analyst_total_tokens,
        "latency_ms": analyst_total_latency_ms,
        "cost_usd": None,
        "decision": baseline_result["decision"],
        "passed": baseline_passed,
    },
    {
        "variant": "改进版本",
        "sources_used": len(fixed_result["received_sources"]),
        "api_calls": 3,
        "total_tokens": analyst_total_tokens + review_metrics["total_tokens"],
        "latency_ms": analyst_total_latency_ms + review_metrics["latency_ms"],
        "cost_usd": None,
        "decision": fixed_result["decision"],
        "passed": fixed_passed,
    },
]

print("版本 | 已用来源 | API 次数 | 总 Token | 顺序等待时间 | 成本 USD | 决定 | 任务成功")
for row in ablation_rows:
    values = [
        row["variant"],
        str(row["sources_used"]),
        str(row["api_calls"]),
        str(row["total_tokens"]),
        f"{row['latency_ms']} ms",
        str(row["cost_usd"]),
        row["decision"],
        str(row["passed"]),
    ]
    print(" | ".join(values))

版本 | 已用来源 | API 次数 | 总 Token | 顺序等待时间 | 成本 USD | 决定 | 任务成功
错误基线 | 1 | 2 | 994 | 12801 ms | None | launch | False
改进版本 | 2 | 3 | 1612 | 20283 ms | None | hold | True


表格显示，基线付出了两次 Analyst 调用，却只使用一个来源并得到错误的 `launch`；改进版本多付出一次 Reviewer 的 Token 和顺序等待时间，换来两个来源都被使用以及正确的 `hold`。金额因 provider 未返回而保持 `None`，表中延迟是本 Notebook 顺序执行三次请求的实测总和，不代表并行部署时的墙钟时间。

## 8.2 总结机制变化

最后只保留加入可靠编排前后的关键状态。模型、任务和 Analyst 回执没有更换，唯一变化是 Harness 何时停止、是否汇总全部来源，以及是否经过 Reviewer。

In [25]:
# 每个字段都连接基线结果与改进结果
# 变化重点是完成条件，而不是模型能力变化
multi_agent_effect = {
    "completion_rule": "first handoff -> all handoffs + reviewer",
    "sources_used": f"{len(baseline_result['received_sources'])} -> {len(fixed_result['received_sources'])}",
    "final_decision": f"{baseline_result['decision']} -> {fixed_result['decision']}",
    "task_success": f"{baseline_passed} -> {fixed_passed}",
}

for name, change in multi_agent_effect.items():
    print(name, "：", change)

completion_rule ： first handoff -> all handoffs + reviewer
sources_used ： 1 -> 2
final_decision ： launch -> hold
task_success ： False -> True


输出中的 `first handoff -> all handoffs + reviewer`、`1 -> 2`、`launch -> hold` 和 `False -> True` 说明：多 Agent 的核心价值不是增加角色数量，而是用显式任务状态、结构化 handoff、完成屏障和独立审阅管理协作。它会增加 Token 与延迟，因此更适合能够拆分且必须汇总多方证据的任务；对于单一步骤任务，单 Agent 通常更简单。至此，本 Notebook 的消融对照结束。

## 8.3 拓展

### nano 版省略了什么

nano 版只有两个 Analyst 和一个 Reviewer 的顺序编排，没有并行调度、角色动态生成、共享状态冲突、通信预算、超时、部分失败、信誉、死锁和跨组织身份。生产多 Agent 系统必须证明分工收益超过额外 Token、延迟与协调错误，并用 trace 保留每次 handoff 的责任边界。

### 延伸阅读


1. 2025, [Google, Announcing the Agent2Agent protocol](https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/)：跨 Agent 任务委派、状态和产物交换。
2. 2025, [Anthropic, How we built our multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system)：并行子 Agent、Lead Agent 汇总与生产权衡。
3. 2025, [Why Do Multi-Agent LLM Systems Fail?](https://arxiv.org/abs/2503.13657)：多 Agent 失败分类、传播路径与诊断框架。